# 07 — Model 2.2 ablations

**What this notebook shows.** Three ablation arms decomposing the class-aware freeze, each one mechanism removed or relocated, run through the shared harness and read against the stored family.

**Arm 1, gate only.** The M1.2 base with the freeze and nothing else: no flag likelihood factors anywhere, fires consumed only as gate triggers,

$$\tau_k(t) = \begin{cases} 0 & \text{after the first bias-class fire homed to } k \\ \tau_k & \text{otherwise} \end{cases}$$

with the update the bare correctness Bayes. Asks whether the dynamics value needs the evidence channel — the third corner of the evidence-by-dynamics square (M1.2 neither, M2.1.1 evidence only, this arm dynamics only, M2.2 both).

**Arm 2, the gate on the state-view chassis.** M2.1.2 verbatim (four-cell joint, flags read B, trap-aware emissions) plus the freeze on the mastery drift of the home KC after a bias-class fire. The habit axis was frozen already; this gates the skill axis too. The fork's other body gets the dynamics test.

**Arm 3, the unratcheted gate.** M2.2 with forgiveness on counter-evidence: a bias-class quiet on the frozen home KC unfreezes it, a later fire refreezes. Tests whether the ratchet's permanence earns its keep.

All arms share the mounted stack, the registered kappa (5), and the class table (bias: conjunction, inverse, time-axis, base-rate neglect; skill: denominator neglect). Each arm refits its own stage two, so comparisons are matched-fold against stored predictions; small pre-fire deltas are refit shifts, not mechanism.

**File layout.**
* The arms: `scripts/model_2_2_ablations.py`
* Stored references: `cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv`, `cache/model_2_1_1/Model_2_1_1/predictions.csv`, `cache/model_2_1_2/Model_2_1_2/predictions.csv`, `cache/model_2_2/Model_2_2/predictions.csv` — none re-run here
* The inner-chain cache: `cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess/`
* Saved outputs: `cache/model_2_2_ablations/<ArmName>/` (cell at the bottom)

**Protocol.** 26-fold leave-one-participant-out, predict-before-update, 312 qc targets, qc base rate 0.641, wrong prevalence 0.359, base-rate log-loss 0.6534.

In [1]:
import pandas as pd
import numpy as np
from scripts.data import load_data
from scripts.evaluator import Evaluator, _metrics
from scripts.model_2_2_ablations import (Model_2_2_Gate_Only, Model_2_2_On_State_View,
                                         Model_2_2_Unratcheted)
from scripts.model_1_2_outer_chain import load_internal_chains

DATA = 'data/data_annotated.csv'
CACHE_DIR = 'cache/model_1_2_internal_chain/Model_1_2_Internal_Slip_And_Guess'
STORED = {
    'M1.2': 'cache/model_1_2_outer_chain/Model_1_2_MIX2/predictions.csv',
    'M2.1.1': 'cache/model_2_1_1/Model_2_1_1/predictions.csv',
    'M2.1.2': 'cache/model_2_1_2/Model_2_1_2/predictions.csv',
    'M2.2': 'cache/model_2_2/Model_2_2/predictions.csv',
}

df = load_data(DATA)
cache = load_internal_chains(CACHE_DIR)
stored = {name: pd.read_csv(path) for name, path in STORED.items()}
print(len(df), 'rows |', len(cache), 'cached folds |', {k: len(v) for k, v in stored.items()})

312 rows | 26 cached folds | {'M1.2': 312, 'M2.1.1': 312, 'M2.1.2': 312, 'M2.2': 312}


## 1. Run
The three arms through the shared harness with the cached inner chains. The family references come from stored predictions, never re-fitted.

In [2]:
kw = dict(n_restarts=3, chain_cache=cache)
arms = {}
for cls in (Model_2_2_Gate_Only, Model_2_2_On_State_View, Model_2_2_Unratcheted):
    arms[cls.__name__] = Evaluator(cls, df, model_kwargs=kw).run()
    print(cls.__name__, 'done |', int(arms[cls.__name__].metrics['n']), 'targets')

Model_2_2_Gate_Only done | 312 targets
Model_2_2_On_State_View done | 312 targets
Model_2_2_Unratcheted done | 312 targets


## 2. Results

### 2.1 Headline metrics, arms beside the stored family

In [3]:
rows = []
for name, p in list(stored.items()) + [(n, e.predictions) for n, e in arms.items()]:
    m = _metrics(p.y_true, p.p_pred)
    rows.append(dict(model=name, auc=round(m['auc'],4), auprc_wrong=round(m['auprc_wrong'],4),
                     bal_acc=round(m['bal_acc'],4), log_loss=round(m['log_loss'],4),
                     accuracy=round(m['accuracy'],4)))
pd.DataFrame(rows).set_index('model')

,auc,auprc_wrong,bal_acc,log_loss,accuracy
model,,,,,
M1.2,0.6741,0.5747,0.6018,0.6070,0.6859
M2.1.1,0.6907,0.5908,0.6073,0.6002,0.6955
M2.1.2,0.6658,0.5651,0.6093,0.6112,0.6955
M2.2,0.7019,0.5918,0.6221,0.5939,0.7019
Model_2_2_Gate_Only,0.6829,0.5858,0.6227,0.6031,0.7051
Model_2_2_On_State_View,0.6729,0.5725,0.6138,0.6077,0.6987
Model_2_2_Unratcheted,0.7015,0.5915,0.6221,0.5943,0.7019


### 2.2 The evidence-by-dynamics square
The designed disagreement set is every row after a student's first bias-class fire (P23 Q3; P02, P03, P11, P20, P24 Q4; P06 Q6; P01 never — denominator is skill-class). The square reads M1.2 (neither channel), M2.1.1 (evidence only), gate-only (dynamics only), M2.2 (both).

In [4]:
first_bias = {'P02': 4, 'P03': 4, 'P06': 6, 'P11': 4, 'P20': 4, 'P23': 3, 'P24': 4}
def split_auc(p):
    q = p.copy()
    q['pf'] = q.apply(lambda r: r.participant_id in first_bias
                      and r.question_number > first_bias[r.participant_id], axis=1)
    return (round(_metrics(q[q.pf].y_true, q[q.pf].p_pred)['auc'], 3),
            round(_metrics(q[~q.pf].y_true, q[~q.pf].p_pred)['auc'], 3),
            int(q.pf.sum()))
rows = []
for name, p in [('M1.2 (neither)', stored['M1.2']),
                ('M2.1.1 (evidence only)', stored['M2.1.1']),
                ('gate only (dynamics only)', arms['Model_2_2_Gate_Only'].predictions),
                ('M2.2 (both)', stored['M2.2'])]:
    d, r, n = split_auc(p)
    rows.append(dict(model=name, designed_set=d, rest=r, n_designed=n))
pd.DataFrame(rows).set_index('model')

,designed_set,rest,n_designed
model,,,
M1.2 (neither),0.519,0.666,55
M2.1.1 (evidence only),0.550,0.665,55
gate only (dynamics only),0.545,0.667,55
M2.2 (both),0.625,0.665,55


### 2.3 The gate on the state-view chassis
Matched-fold contrast against stored M2.1.2: does the freeze help the fork's other body?

In [5]:
j = arms['Model_2_2_On_State_View'].predictions.merge(
    stored['M2.1.2'], on=['participant_id','question_number'], suffixes=('_g','_b'))
j['good'] = np.where(j.y_true_g == 1, j.p_pred_g - j.p_pred_b, j.p_pred_b - j.p_pred_g)
a_g = _metrics(j.y_true_g, j.p_pred_g); a_b = _metrics(j.y_true_b, j.p_pred_b)
print(f'gated state view auc {a_g["auc"]:.4f} | chassis {a_b["auc"]:.4f}')
print(f'rows helped (good > 0.01): {int((j.good > 0.01).sum())} | hurt: {int((j.good < -0.01).sum())}')
d, r, n = split_auc(arms['Model_2_2_On_State_View'].predictions)
print(f'designed set {d} | rest {r}')

gated state view auc 0.6729 | chassis 0.6658
rows helped (good > 0.01): 17 | hurt: 6
designed set 0.53 | rest 0.656


### 2.4 The unratcheted gate
Matched against stored M2.2: does forgiveness on counter-evidence change anything?

In [6]:
k = arms['Model_2_2_Unratcheted'].predictions.merge(
    stored['M2.2'], on=['participant_id','question_number'], suffixes=('_u','_r'))
moved = (k.p_pred_u - k.p_pred_r).abs() > 0.005
a_u = _metrics(k.y_true_u, k.p_pred_u); a_r = _metrics(k.y_true_r, k.p_pred_r)
print(f'unratcheted auc {a_u["auc"]:.4f} | M2.2 {a_r["auc"]:.4f}')
print(f'rows moved > 0.005: {int(moved.sum())} of {len(k)}')
d, r, n = split_auc(arms['Model_2_2_Unratcheted'].predictions)
print(f'designed set {d} | rest {r}')

unratcheted auc 0.7015 | M2.2 0.7019
rows moved > 0.005: 9 of 312
designed set 0.624 | rest 0.665


## 3. Conclusion

* **The two channels are super-additive, and the interaction carries the thesis claim.** On the designed set, evidence alone buys +0.036 over M1.2 (0.554 vs 0.518), the gate alone +0.027 (0.545), their sum 0.063, and the combination delivers +0.107 (0.625). The fire's likelihood-crash gives the freeze something to hold down; the freeze makes the crash persist instead of refilling. Neither channel suffices alone: typed identity matters specifically in time, through the interaction of loud evidence with class-aware persistence.
* **The gate helps the state-view chassis too, but far less, and the asymmetry explains the synergy from the other side.** Gated M2.1.2 lands at ~0.673 against its chassis's ~0.666 (19 rows helped, 6 hurt; designed set +0.014 against the observation chassis's +0.071). The state view's fires heat B while mastery stays high, so the freeze holds a ceiling-level belief — the gate needs a crash to preserve.
* **The ratchet is empirically inert at this n.** Unratcheted 0.7019 against M2.2's 0.7018; 11 of 312 rows move above 0.005; designed set identical to the third decimal. The unfreezing quiets exist (the quintet's Q6) but land where kc2 cell evidence had already restored the ceiling, so the returned optimism had nothing to add. Permanence versus forgiveness-on-quiet is a stylistic choice the data cannot distinguish here; the registered hard ratchet stands, and the claim that the ratchet matters is explicitly not made.
* **A threshold-metric footnote.** Gate-only posts the family's best balanced accuracy (~0.623) and accuracy (~0.705): the pure freeze is a good classifier even while the full model is the better forecaster.
* **Caveats.** Each arm refits its own stage two, so small pre-fire deltas are refit shifts; the designed set is 55 rows; participant-clustered intervals remain deferred until the family is complete.

## 4. Save
Persist each arm: per-fold bridge, shape, and whatever fitted tables the arm carries, plus predictions, metrics, and the index.

In [8]:
import os
import json
from scripts.model_2_2_ablations import save_ablation_from_evaluator

out_dir = 'cache/model_2_2_ablations'
for name, ev in arms.items():
    print(save_ablation_from_evaluator(ev, out_dir))

cache/model_2_2_ablations/Model_2_2_Gate_Only
cache/model_2_2_ablations/Model_2_2_On_State_View
cache/model_2_2_ablations/Model_2_2_Unratcheted
